# Airspace Object Detection: Combined Model (MBAOD + AOD 4)

Trains one YOLO11s on **both** datasets together, then tests it on **each** test set separately. The question: does mixing data sources close the generalization gap we found? (The MBAOD-only model dropped to drone AP50-95 0.284 on AOD 4.)

**Classes (6):** airplane, bird, helicopter, kite, parachute, drone. MBAOD's `uav` and AOD 4's `drone` are merged into one `drone` class.

**How to run:** use **Save Version → Save & Run All (Commit)** with GPU T4 x2, Internet on, and the `ROBOFLOW_API_KEY` secret attached. It runs in the background for about 4 to 5 hours, and the results land in the version's Output tab.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# 1. Download both datasets in YOLO format
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
mbaod = rf.workspace("fmk-mwmyi").project("mbaod").version(1).download("yolov8", location="/kaggle/working/mbaod")
aod4  = rf.workspace("airborne-object-detection").project("airborne-object-detection-4-aod4").version(6).download("yolov8", location="/kaggle/working/aod4")

In [ ]:
# 2. Shared class list: MBAOD's order, with 'uav' renamed to 'drone'
import yaml, os, glob
mb_names  = yaml.safe_load(open(os.path.join(mbaod.location, "data.yaml")))["names"]
aod_names = yaml.safe_load(open(os.path.join(aod4.location, "data.yaml")))["names"]
names = ["drone" if n == "uav" else n for n in mb_names]
id_map = {i: names.index(n) for i, n in enumerate(aod_names)}
print("Shared classes:", names)
print("AOD 4 -> shared ids:", {aod_names[k]: names[v] for k, v in id_map.items()})

In [ ]:
# 3. Rewrite AOD 4 labels into the shared ids, for every split.
#    Images are linked, not copied, so this takes seconds and no extra disk.
#    YOLO finds each label by swapping 'images' for 'labels' in the image path.
# FIX: Ultralytics resolves symlinked image folders back to the original labels,
#      so rewrite the AOD 4 labels in place instead.
REMAP = aod4.location
for split in ["train", "valid", "test"]:
    img_link = f"{REMAP}/{split}/images"
    os.makedirs(f"{REMAP}/{split}/labels", exist_ok=True)
    if not os.path.exists(img_link):
        os.symlink(os.path.join(aod4.location, split, "images"), img_link)
    for f in glob.glob(os.path.join(aod4.location, split, "labels", "*.txt")):
        out = []
        for line in open(f):
            p = line.split()
            if p:
                out.append(" ".join([str(id_map[int(p[0])])] + p[1:]))
        open(f"{REMAP}/{split}/labels/{os.path.basename(f)}", "w").write("\n".join(out))
    print(split, len(os.listdir(f"{REMAP}/{split}/labels")), "label files")

In [ ]:
# 4. Dataset configs: one combined set for training, and one per test set
def write_yaml(path, train, val, test):
    yaml.safe_dump({"train": train, "val": val, "test": test, "names": names}, open(path, "w"))
    return path

MB = mbaod.location
combined_yaml = write_yaml("/kaggle/working/combined.yaml",
    [f"{MB}/train/images", f"{REMAP}/train/images"],
    [f"{MB}/valid/images", f"{REMAP}/valid/images"],
    [f"{MB}/test/images",  f"{REMAP}/test/images"])
mbaod_test_yaml = write_yaml("/kaggle/working/mbaod_test.yaml",
    f"{MB}/train/images", f"{MB}/valid/images", f"{MB}/test/images")
aod4_test_yaml = write_yaml("/kaggle/working/aod4_test.yaml",
    f"{REMAP}/train/images", f"{REMAP}/valid/images", f"{REMAP}/test/images")

n_train = len(glob.glob(f"{MB}/train/images/*")) + len(glob.glob(f"{REMAP}/train/images/*"))
print("Combined training images:", n_train)

In [ ]:
# 5. Train on the combined data (same settings as before, 35 epochs)
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data=combined_yaml, epochs=35, imgsz=640, batch=32, device=[0, 1],
            project="/kaggle/working/runs", name="combined_640", patience=10)

In [ ]:
# 6. Test the combined model on each test set separately
best = YOLO("/kaggle/working/runs/combined_640/weights/best.pt")
results = {}
for label, cfg in [("MBAOD test", mbaod_test_yaml), ("AOD 4 test", aod4_test_yaml)]:
    m = best.val(data=cfg, split="test", imgsz=640, plots=True,
                 project="/kaggle/working/runs", name="combined_on_" + label.split()[0].lower().replace(" ", ""))
    results[label] = {"mAP50": m.box.map50, "mAP50-95": m.box.map,
                      **{n: m.box.maps[i] for i, n in enumerate(names)}}
    print(f"\n{label}: mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}")
    for i, n in enumerate(names):
        print(f"  {n:12s} AP50-95 = {m.box.maps[i]:.3f}")

In [ ]:
# 7. Side-by-side with the earlier models (AP50-95)
import json
earlier = {
    "AOD 4 model on AOD 4":    {"mAP50-95": 0.680, "drone": 0.630, "bird": 0.717, "airplane": 0.735, "helicopter": 0.641},
    "MBAOD model on MBAOD":    {"mAP50-95": 0.771, "drone": 0.680, "bird": 0.748, "airplane": 0.813, "helicopter": 0.772},
    "MBAOD model on AOD 4":    {"mAP50-95": 0.467, "drone": 0.284, "bird": 0.345, "airplane": 0.712, "helicopter": 0.525},
}
rows = dict(earlier)
rows["Combined model on MBAOD"] = results["MBAOD test"]
rows["Combined model on AOD 4"] = results["AOD 4 test"]
cols = ["mAP50-95", "drone", "bird", "airplane", "helicopter"]
print(f"{'':28s}" + "".join(f"{c:>12s}" for c in cols))
for k, v in rows.items():
    print(f"{k:28s}" + "".join(f"{v[c]:12.3f}" for c in cols))
json.dump({k: {c: float(v[c]) for c in v} for k, v in rows.items()},
          open("/kaggle/working/results_summary.json", "w"), indent=2)

In [ ]:
# 8. Confusion matrices for both test sets
from IPython.display import Image, display
for d in ["combined_on_mbaod", "combined_on_aod"]:
    p = f"/kaggle/working/runs/{d}/confusion_matrix_normalized.png"
    if os.path.exists(p):
        print(d); display(Image(p, width=650))

In [ ]:
# 9. Keep the outputs small: zip the runs, then delete the downloaded datasets
import shutil
shutil.make_archive("/kaggle/working/combined_results", "zip", "/kaggle/working/runs")
for d in ["/kaggle/working/mbaod", "/kaggle/working/aod4", REMAP]:
    shutil.rmtree(d, ignore_errors=True)
print(os.listdir("/kaggle/working"))

## How to read the result

- **Combined model on AOD 4, drone column:** this is the key number. If it climbs from 0.284 back toward the AOD 4-only model's 0.630, mixing data sources fixes most of the gap.
- **Combined model on MBAOD:** check that it hasn't dropped much below 0.771. Mixing data shouldn't make the model worse at either dataset.
- If both hold, this is your **final demo model**. Download `combined_results.zip` from the Output tab.